In [1]:
from pyspark.sql import SparkSession
from awsglue.context import GlueContext

In [2]:
spark = SparkSession.builder \
    .appName("S3WriteTest") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localstack:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

glueContext = GlueContext(spark.sparkContext)

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getO

In [3]:
db_url = "jdbc:postgresql://postgre_localdb:5432/postgres_db"
db_table = "customers"
db_properties = {
    "user": "myusers",
    "password": "passwords",
    "driver": "org.postgresql.Driver"
}

try:
    df = spark.read.jdbc(url=db_url, table=db_table, properties=db_properties)
    df.show()
except Exception as e:
    print("Error reading from PostgreSQL:", e)

+-----------+----------+---------+--------------------+--------------------+--------------------+---------+-------------------+-------------------+
|customer_id|first_name|last_name|               email|        phone_number|     current_address|is_active|         created_at|    last_updated_at|
+-----------+----------+---------+--------------------+--------------------+--------------------+---------+-------------------+-------------------+
|          1|     Julia|   Cooper|rpetersen@example...|   768.508.9477x8717|3082 Joshua Manor...|     true|2024-11-24 01:29:24|2022-09-03 04:12:33|
|          2|   Cynthia|  Ramirez|teresa47@example.net|     +1-691-563-5068|9353 Rojas Corner...|     true|2020-11-10 00:54:27|2021-01-19 19:14:44|
|          3|   Matthew|Alexander|larsonkristen@exa...|        742.206.6778|129 Townsend Corn...|     true|2021-06-06 14:05:55|2021-03-13 21:14:27|
|          4|     Tammy|  Daniels|rebecca39@example...|        568.782.7157|4907 Cooke Burgs ...|     true|2024-

In [5]:
df.show()

+-----------+----------+---------+--------------------+--------------------+--------------------+---------+-------------------+-------------------+
|customer_id|first_name|last_name|               email|        phone_number|     current_address|is_active|         created_at|    last_updated_at|
+-----------+----------+---------+--------------------+--------------------+--------------------+---------+-------------------+-------------------+
|          1|     Julia|   Cooper|rpetersen@example...|   768.508.9477x8717|3082 Joshua Manor...|     true|2024-11-24 01:29:24|2022-09-03 04:12:33|
|          2|   Cynthia|  Ramirez|teresa47@example.net|     +1-691-563-5068|9353 Rojas Corner...|     true|2020-11-10 00:54:27|2021-01-19 19:14:44|
|          3|   Matthew|Alexander|larsonkristen@exa...|        742.206.6778|129 Townsend Corn...|     true|2021-06-06 14:05:55|2021-03-13 21:14:27|
|          4|     Tammy|  Daniels|rebecca39@example...|        568.782.7157|4907 Cooke Burgs ...|     true|2024-


1) Moment 01 -> Full_refresh table
2) Moment 02 -> Sync Data
             -> A) Insert -> Only Append
             -> B) Delete -> How to camputure? Is it necessary process? Why in what scenarios?
             -> C) Update -> Deal with the old partition (migrate the record to new partition)

- Create partition year, month, day
- 



In [14]:
df.select(col("last_updated_at"),month("last_updated_at").alias("date")) \
  .show()

+-------------------+----+
|    last_updated_at|date|
+-------------------+----+
|2022-09-03 04:12:33|   9|
|2021-01-19 19:14:44|   1|
|2021-03-13 21:14:27|   3|
|2023-12-19 07:58:44|  12|
|2021-09-18 01:37:18|   9|
|2023-12-17 09:26:33|  12|
|2020-03-07 06:03:10|   3|
|2022-03-05 14:17:52|   3|
|2025-02-10 15:59:53|   2|
|2025-08-06 22:45:00|   8|
|2020-09-28 19:34:42|   9|
|2023-07-29 02:20:43|   7|
|2023-12-30 23:28:09|  12|
|2021-03-29 11:55:20|   3|
|2021-07-16 13:15:30|   7|
|2022-01-05 11:51:58|   1|
|2023-10-18 23:14:58|  10|
|2024-12-03 10:45:01|  12|
|2020-06-01 06:41:57|   6|
|2023-03-20 14:58:50|   3|
+-------------------+----+
only showing top 20 rows



In [35]:
def create_partition_date(df, source_data_partition:str):
    df = df.withColumn("partition_date", to_date(df[f"{source_data_partition}"], "yyyy-MM-dd"))
    df = df.withColumn("year", year(df["partition_date"]))
    df = df.withColumn("month", month(df["partition_date"]))
    df = df.withColumn("day", dayofmonth(df["partition_date"]))
    return df
    

In [36]:
df = create_partition_date(df, "last_updated_at")

In [38]:
df.show()

+-----------+----------+---------+--------------------+--------------------+--------------------+---------+-------------------+-------------------+--------------+----+-----+---+
|customer_id|first_name|last_name|               email|        phone_number|     current_address|is_active|         created_at|    last_updated_at|partition_date|year|month|day|
+-----------+----------+---------+--------------------+--------------------+--------------------+---------+-------------------+-------------------+--------------+----+-----+---+
|          1|     Julia|   Cooper|rpetersen@example...|   768.508.9477x8717|3082 Joshua Manor...|     true|2024-11-24 01:29:24|2022-09-03 04:12:33|    2022-09-03|2022|    9|  3|
|          2|   Cynthia|  Ramirez|teresa47@example.net|     +1-691-563-5068|9353 Rojas Corner...|     true|2020-11-10 00:54:27|2021-01-19 19:14:44|    2021-01-19|2021|    1| 19|
|          3|   Matthew|Alexander|larsonkristen@exa...|        742.206.6778|129 Townsend Corn...|     true|202

In [42]:
def save_data_s3(df, s3_path:str, mode:str)-> None:
    df.write\
      .partitionBy("year", "month")\
      .mode(mode)\
      .parquet(s3_path_target)

In [43]:
s3_path_target = "s3a://brozen-data-lake-bucket/postgres/customer"
save_data_s3(df, s3_path_target, "overwrite")

In [61]:
db_url = "jdbc:postgresql://postgre_localdb:5432/postgres_db" # information_schema
#db_url = "jdbc:postgresql://postgre_localdb:5432/information_schema"
db_table = "information_schema.tables"# "customers"
db_properties = {
    "user": "myusers",
    "password": "passwords",
    "driver": "org.postgresql.Driver"
}

def refresh_table_pipeline(mode:str, table_target:[None, str] = None, spark=spark, db_url:str=db_url,db_properties:str=db_properties):
    
    MODE_OPTIONS = ["full", "partial"]
    if mode not in MODE_OPTIONS:
         raise Exception(f"ERROR: input parameter '{mode}' is not in {MODE_OPTIONS}, please check it out!")
    
    if mode == "full":
        print("Start FULL Pipeline")
        db_table = """(SELECT table_name FROM information_schema.tables WHERE table_schema = 'public') AS talbe_name_list """
        table_list = spark_connection_db_query(spark, db_url, db_table, db_properties)
        table_list.show()


    if mode == "partial":
        if table_target:
            print("Start PARTIAL Pipeline")
            table_list = spark_connection_db_query(spark, db_url, table_target, db_properties)
            table_list.show()
        
        else:
            raise Exception(f"ERROR: table_target not informed to full refresh!")



def spark_connection_db_query(spark, db_url:str, db_table:str, db_properties:str):
    try:
        df = spark.read.jdbc(url=db_url, table=db_table, properties=db_properties)
        return df 
    except Exception as e:
        print("Error reading from PostgreSQL:", e)
        raise    

In [63]:
refresh_table_pipeline("full")

Start FULL Pipeline
+------------+
|  table_name|
+------------+
|   customers|
|    accounts|
|transactions|
+------------+



In [ ]:
-> QUERY the SQL -> threatment -> persist

In [58]:
db_url = "jdbc:postgresql://postgre_localdb:5432/postgres_db" # information_schema
#db_url = "jdbc:postgresql://postgre_localdb:5432/information_schema"
db_table = "information_schema.tables"# "customers"
db_properties = {
    "user": "myusers",
    "password": "passwords",
    "driver": "org.postgresql.Driver"
}

try:
    df = spark.read.jdbc(url=db_url, table=db_table, properties=db_properties)
    df.show()
except Exception as e:
    print("Error reading from PostgreSQL:", e)

+-------------+------------+--------------------+----------+----------------------------+--------------------+-------------------------+------------------------+----------------------+------------------+--------+-------------+
|table_catalog|table_schema|          table_name|table_type|self_referencing_column_name|reference_generation|user_defined_type_catalog|user_defined_type_schema|user_defined_type_name|is_insertable_into|is_typed|commit_action|
+-------------+------------+--------------------+----------+----------------------------+--------------------+-------------------------+------------------------+----------------------+------------------+--------+-------------+
|  postgres_db|      public|           customers|BASE TABLE|                        null|                null|                     null|                    null|                  null|               YES|      NO|         null|
|  postgres_db|      public|            accounts|BASE TABLE|                        null|   

In [ ]:
(select table_name from information_schema.tables
where table_schema = 'public') AS "talbe_name_list"


In [29]:
print(type(df))

<class 'pyspark.sql.dataframe.DataFrame'>


In [31]:
dir(df)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_collect_as_arrow',
 '_jcols',
 '_jdf',
 '_jmap',
 '_joinAsOf',
 '_jseq',
 '_lazy_rdd',
 '_repr_html_',
 '_sc',
 '_schema',
 '_session',
 '_sort_cols',
 '_sql_ctx',
 '_support_repr_html',
 '_to_corrected_pandas_type',
 'agg',
 'alias',
 'approxQuantile',
 'cache',
 'checkpoint',
 'coalesce',
 'colRegex',
 'collect',
 'columns',
 'corr',
 'count',
 'cov',
 'createGlobalTempView',
 'createOrReplaceGlobalTempView',
 'createOrReplaceTempView',
 'createTempView',
 'crossJoin',
 'crosstab',
 'cube',
 'describe',
 'distinct',
 'drop',
 'dropDuplicates',
 'drop_duplicates',
 'dropna',
 'dtypes',
 

In [25]:
from pyspark.sql.functions import col, to_date, month, year, dayofmonth 

In [ ]:
a

In [5]:
s3_bucket = "brozen-data-lake-bucket"
print(f"Listing contents of S3 bucket: {s3_bucket}")
try:

    GlueContext.create_dynamic_frame.from_options(
        connection_type="s3",
        connection_options={"paths": [s3_bucket]},
        format="csv"
    )
    print("S3 connection successful!")
except Exception as e:
    print(f"S3 connection failed: {e}")

Listing contents of S3 bucket: brozen-data-lake-bucket
S3 connection successful!


25/10/04 19:29:08 WARN HadoopDataSource: Skipping Partition {} as no new files detected @ brozen-data-lake-bucket or path does not exist
